In [2]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import pickle
from tensorflow.keras.models import load_model
from collections import Counter
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

import fasttext
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

I0000 00:00:1781551977.371136   30347 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781551977.436553   30347 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781551979.138107   30347 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/jax/Downloads/complaints/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
id2label = {0: "Delivery Issue", 1: "Food Quality", 2: "Hygiene", 3: "Service Quality", 
            4: "Pricing", 5: "Order Accuracy"}
S_id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}
E_id2label = {0: "Frustrated", 1: "Satisfied", 2: "Disgusted", 3: "Neutral"}

In [4]:
def predict (model_location: str, embedded_text, type = "S", for_ML = True, prob = False):
    """
     Docstring for predict function: 
        1. Load the pretrained model whether it's ML or DL (GRU, Bi-LSTM only).
        2. Checking the right classification type (emotion, sentiment or problem type).
        3. Convert the predcited class id into the approperate label.

    Params:
        model_location: the location of the trained ML or DL model (str).
        embedded_text: the incoming feedback or comment after vectorization or embedding.
        type: the type of classification ("S" for sentiment, "E" for emotion and "P" for problem type) (str).
        for_ML: if you want to predict using either ML "True" or DL (GRU, Bi-LSTM only) "False" (bool).
        prob: if you want to predict the probabilities of the classes (bool, default "False").

    Return Values:
        The final prediction label or labels' probabilities.
    """
    # Choosing whether to use ML or DL (GRU, Bi-LSTM only)
    if for_ML:
        with open(model_location, "rb") as f:
            model = pickle.load(f)
        if prob:
            pred_probs = model.predict_proba(embedded_text)
            return pred_probs
        else:
            pred_id = int(model.predict(embedded_text)[0])
    else:
        model = load_model(model_location)
        probs = model.predict(embedded_text)
        if prob:
            return probs
        else:
            # convert probabilities → class id
            pred_id = int(np.argmax(probs, axis=1)[0])

    # Choosing the correct the classifciation type whether it's (sentiment, emotion or problem type)
    if type == "S":
        return S_id2label[pred_id]
    elif type == "E":
        return E_id2label[pred_id]
    elif type == "P":
        return id2label[pred_id]
    else:
        raise ValueError("Value must be either S for sentiment prediction, E for emotion prediction or P for problem type prediction")


In [5]:


# If there is a GPU use it otherwise use a CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
# Load trained bert model
bert = AutoModelForSequenceClassification.from_pretrained(r"/home/jax/Downloads/complaints/backend/models/AR_Models/AR/Probelm-Type-Classification/arabert_67").to(device)
# Load the trained tokenizer
tokenizer = AutoTokenizer.from_pretrained(r"/home/jax/Downloads/complaints/backend/models/AR_Models/AR/Probelm-Type-Classification/arabert_67")
bert.eval()

def pred_b(text : str, prob = False) -> str:
    """ 
    Docstring for pred_b function: 
        1. Load the pretrained bert model.
        2. Load the pretrained tokenizer.
        3. Tokenize the input text.
        4. Predict the output label.
        5. Map the label using a mapping dictionary

    Params:
        text: the incoming feedback or comment (str)
        prob: if you want to predict the probabilities of the classes (bool, default "False").

    Return Values:
        The predicted value or probabilities
    """
    # Tokenize the text
    inputs = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    return_tensors="pt")
    
    # Move the tokens to the same device as the model
    inputs = {k:v.to(device) for k,v in inputs.items()}

    with torch.no_grad():
        # Unpack the dictonary into the model and predict
        outputs = bert(**inputs)
        # Get the logits (value between -1: 1)
        logits = outputs.logits
        if prob:
            # Get the probabilites of predicted classes
            probs = torch.softmax(logits, dim=-1)
            probs = probs.detach().cpu().numpy()
            return probs
        else:
            # Get the class id with highest value
            pred_id = torch.argmax(logits, dim=1).cpu().numpy()
            pred_id = int(pred_id)
            return id2label[pred_id]


/home/jax/Downloads/complaints/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 302: Error loading CUDA libraries. GPU will not be used. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [6]:

model = fasttext.load_model("/home/jax/Downloads/complaints/backend/models/AR_Models/fasttext_model_Ara.bin")

max_len = 65
# Change the location of the tokenizer if you have to
with open(r"/home/jax/Downloads/complaints/backend/models/AR_Models/DL_tokenizer_ara.pkl", "rb") as f:
    tokenizer = pickle.load(f)
bert_model = AutoModelForSequenceClassification.from_pretrained(r"/home/jax/Downloads/complaints/backend/models/AR_Models/AR/Probelm-Type-Classification/arabert_67").to(device)
ar_tokenizer = AutoTokenizer.from_pretrained(r"/home/jax/Downloads/complaints/backend/models/AR_Models/AR/Probelm-Type-Classification/arabert_67")
def embedd(text: str, for_ML = True):
    """ 
    Docstring for embedd function: 
        1. Load the pretrained fasttext model.
        2. Load the pretrained tokenizer used with GRU & Bi-LSTM only.
        3. If for ML prediction get the embedding vector right away.
        4. If for deep learning prediction (GRU, Bi-LSTM only), tokenize the input text & pad the sequence.

    Params:
        text: the incoming cleaned/preprocessed feedback or comment (str).
        for_ML: if you want to use the embedding for ML prediction "True" or Deep learning "False" (GRU, Bi-LSTM only) (bool).

    Return Values:
        The embedded text ready for prediction.
    """
    if for_ML:
        return np.array(model.get_sentence_vector(text)).reshape(1, -1)
    else:
        inputs = ar_tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

        with torch.no_grad():
            outputs = bert_model.bert(**inputs)

        hidden_states = outputs.last_hidden_state

    # Mean pooling
    embedding = hidden_states.mean(dim=1)

    return embedding.cpu().numpy()

In [7]:
def vote(text: str, weights: list, models: list, soft = False, num_models = 5, clf_type = "P"):
    """
    Docstring for vote function:
        1. Ensample vote using 3 models (1 ML model, 1 DL model and 1 RoBERTa model) using hard or weighted soft voting.
        2. Ensample vote using 5 models (3 ML models, 1 BERT model, 1 RoBERTa model) using hard or weighted soft voting.

    Params:
        text: the incoming cleaned feedback text (str).
        weights: a list of weights whether they are integer, float when using weighted soft voting (List).
        models: a list of the machine learning and deep learning models' locations only not RoBERTa & BERT models (List).
        soft: If you want to use weighted soft voting set the value to "True" otherwise leave it as the defualt value "False" (bool).
        num_models: how many models do you want to use for voting 3 models (1 ML model, 1 DL model and 1 RoBERTa model) or 5 (defualt value) models
            (3 ML models, 1 BERT model, 1 RoBERTa model) in this exact order so you can put the models correctly (int).
        clf_type: what type of classification problem "P" for problem type, "S" for sentiment and "E" for emotion classification (str).
    
    Return Values:
        The final ensampled voted output whether using hard voting or soft voting.
    
    """

    # Embedd cleaned text for machine learning prediction
    embedded_text_M = embedd(text = text)
    # Embedd cleaned text for deep learning (GRU, Bi-LSTM only) prediction
    embedded_text_D = embedd(text = text, for_ML = False)

    if num_models == 3:
        if soft:
            # Weighted soft voting
            # ML prediction
            probs1 = predict(model_location=models[0], for_ML = True, prob= True, embedded_text = embedded_text_M, type = clf_type)
            # DL prediction
            probs2 = predict(model_location=models[1], for_ML= False, prob = True, embedded_text = embedded_text_D, type = clf_type)
            # RoBERTa prediction
            probs3 = pred_b(text, prob = True)
            w1 = weights[0]
            w2 = weights[1]
            w3 = weights[2]

            weighted_probs = (
                w1 * probs1 +
                w2 * probs2 +
                w3 * probs3 ) / (w1 + w2 + w3)

            # Get the index of the highest probability
            final_pred_weighted = np.argmax(weighted_probs, axis=1)

            if clf_type == "P":
                return id2label[final_pred_weighted[0]]
            elif clf_type == "S":
                return S_id2label[final_pred_weighted[0]]
            elif clf_type == "E":
                return E_id2label[final_pred_weighted[0]]

        if not soft:
            # Hard voting
            # ML prediction
            label1 = predict(model_location=models[0], for_ML = True, prob= False, embedded_text = embedded_text_M, type = clf_type)
            # DL prediction
            label2 = predict(model_location=models[1], for_ML= False, prob = False, embedded_text = embedded_text_D, type = clf_type)
            # RoBERTa prediction
            label3 = pred_b(text, prob = False)

            preds = [label1, label2, label3, label4, label5]
            return Counter(preds).most_common(1)[0][0]
        
    if num_models == 5:
        if soft:
            # Weighted soft voting
            # ML prediction
            probs1 = predict(model_location = models[0], for_ML = True, prob = True, embedded_text = embedded_text_M, type = clf_type)
            probs2 = predict(model_location= models[1], for_ML= True, prob = True, embedded_text = embedded_text_M, type = clf_type)
            probs3 = predict(model_location= models[2], for_ML=True, prob = True, embedded_text = embedded_text_M, type = clf_type)
            # BERT prediction
            probs4 = pred_b(text = text, prob= True)
            # RoBERTa prediction
            probs5 = pred_b(text = text, prob= True)
            
            w1 = weights[0]
            w2 = weights[1]
            w3 = weights[2]
            w4 = weights[3]
            w5 = weights[4]

            weighted_probs = (
                w1 * probs1 +
                w2 * probs2 +
                w3 * probs3 +
                w4 * probs4 +
                w5 * probs5
            ) / (w1 + w2 + w3 + w4 + w5)

            # Get the index of the highest probability
            final_pred_weighted = np.argmax(weighted_probs, axis=1)

            if clf_type == "P":
                return id2label[final_pred_weighted[0]]
            elif clf_type == "S":
                return S_id2label[final_pred_weighted[0]]
            elif clf_type == "E":
                return E_id2label[final_pred_weighted[0]]
            
        if not soft:
            # Hard voting
            label1 = predict(model_location= models[0], for_ML= True, prob = False, embedded_text = embedded_text_M, type = clf_type)
            print(label1)
            label2 = predict(model_location= models[1], for_ML= True, prob = False, embedded_text = embedded_text_M, type = clf_type)
            print(label2)
            label3 = predict(model_location= models[2], for_ML= True, prob = False, embedded_text = embedded_text_M, type = clf_type)
            print(label3)
            # BERT prediction
            label4 = pred_b(text = text)
            print(label4)
            # RoBERTa prediction
            label5 = pred_b(text = text)
            print(label5)

            preds = [label1, label2, label3, label4, label5]
            return Counter(preds).most_common(1)[0][0]

In [8]:
import sys,os

# Get current working directory in notebook
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

In [9]:
from app.config import settings

In [ ]:
models =[settings.AR_PROBLEM_LR_F_PATH,
        settings.AR_PROBLEM_SVM_A_PATH,
        settings.AR_PROBLEM_LR_A_PATH

]

: 

In [ ]:
print(vote(text = "الطعام بارد جدا", weights = [1, 1, 1, 1, 1], models = models, soft = False, num_models = 5, clf_type = "P"))

In [ ]:
import sys
import numpy
import fasttext

print(f"Python version: {sys.version}")
print(f"NumPy version: {numpy.__version__}")
print(f"fasttext version: {fasttext.__version__ if hasattr(fasttext, '__version__') else 'Unknown'}")

Python version: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
NumPy version: 1.26.4
fasttext version: Unknown
